# 01 — Data Exploration: EVRealDrive-ES

This notebook reproduces the feature distributions shown in **Figure 1** of the paper:

- Distance (km)
- Temperature (°C)
- Road inclination (rad)
- Time increment (s)
- SoC drop (%)

It also reports summary statistics (mean, std) and checks the train/val/test split for leakage.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make src/ importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

DATA_DIR = Path('../data')
FIG_DIR  = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)
print('Data dir:', DATA_DIR.resolve())

## 1. Load derived features

The file `derived_features_sample.csv` contains one row per road segment with the four LSTM input features and the target SoC drop.

Columns:
- `trip_id`      : identifier of the trip
- `segment_id`   : sequential index within the trip
- `distance_km`  : segment distance (km)
- `temperature_c`: ambient temperature (°C)
- `inclination_rad`: road inclination (rad)
- `time_increment_s`: traversal time (s)
- `soc_drop_pp`  : SoC drop (percentage points)

In [ ]:
csv_path = DATA_DIR / 'derived_features_sample.csv'
df = pd.read_csv(csv_path)
print('Rows:', len(df))
print('Trips:', df['trip_id'].nunique())
df.head()

## 2. Summary statistics

These are the numbers quoted in the paper:

- Distance: μ = 89.72 km, σ = 46.31 km
- Temperature: μ = 15.62 °C, σ = 10.23 °C
- Inclination: μ ≈ 0.0003 rad, σ = 0.0476 rad
- Time increment: μ = 10.28 s, σ = 11.43 s
- SoC drop: μ = 1.95 %, σ = 2.28 %

In [ ]:
features = ['distance_km', 'temperature_c', 'inclination_rad',
            'time_increment_s', 'soc_drop_pp']

stats = df[features].agg(['mean', 'std', 'min', 'max']).T
stats.columns = ['mean', 'std', 'min', 'max']
stats.round(4)

## 3. Figure 1 — Feature distributions

The temperature axis spans [-20, 50] °C to match the figure in the paper.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()

panels = [
    ('distance_km',       'Distance (km)',         np.arange(0, 351, 10)),
    ('temperature_c',     'Temperature (°C)',      np.arange(-20, 51, 1)),
    ('inclination_rad',   'Road Inclination (rad)',np.linspace(-0.35, 0.35, 50)),
    ('time_increment_s',  'Time Increment (s)',    np.arange(0, 251, 5)),
    ('soc_drop_pp',       'SoC Drop (%)',          np.arange(0, 21, 0.5)),
]

for ax, (col, title, bins) in zip(axes, panels):
    mu, sigma = df[col].mean(), df[col].std()
    ax.hist(df[col], bins=bins, color='#2b7bba', edgecolor='white', linewidth=0.3)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency')
    ax.text(0.97, 0.95,
            f'μ = {mu:.2f}\nσ = {sigma:.2f}',
            transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7'))

# Hide the unused 6th subplot
axes[-1].axis('off')

plt.suptitle('Dataset Feature Distributions — EVRealDrive-ES',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'dataset_hist.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', FIG_DIR / 'dataset_hist.png')

## 4. Split integrity check

Verify that no `trip_id` appears in more than one split. The paper states that the split is performed **at the trip level**.

In [ ]:
if 'split' in df.columns:
    trip_splits = df.groupby('trip_id')['split'].nunique()
    leaking = trip_splits[trip_splits > 1]
    print(f'Trips present in more than one split: {len(leaking)}')
    if len(leaking) > 0:
        print(leaking)
    else:
        print('OK — no trip-level leakage.')
else:
    print('No split column; assume the sample CSV is one split only.')

## 5. Correlation matrix

Quick sanity check on feature-target relationships.

In [ ]:
corr = df[features].corr().round(3)
corr

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(features))); ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_yticks(range(len(features))); ax.set_yticklabels(features)
for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_correlation.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Takeaways

- SoC drop is right-skewed, consistent with intermittent high-consumption events (steep climbs, cold-weather heating).
- Temperature spans the full [-20, 50] °C range, matching the figure caption.
- No trip-level leakage between splits.
- Distance and time increment dominate the correlation with SoC drop; inclination contributes a smaller but non-negligible signal.